In [ ]:
import cv2
import numpy as np


def stain_features(image: np.ndarray, window: int | None = None, k: int | None = None,
                   dark_thresh: float = 0.08, min_redness: float = 1.8) -> dict:
    """Measure how blotchy the paper background is.
    Returns a dict of scores used by is_stain() plus the stain mask (full resolution)."""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if image.ndim == 3 else image
    h, w = gray.shape

    # 1. Paper background: dilate to erase ink / bleed-through, median blur to smooth
    if window is None:
        window = max(3, int(w * 0.04) | 1)
    if k is None:
        k = max(3, int(w * 0.006) | 1)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    background = cv2.medianBlur(cv2.dilate(gray, kernel), window)

    # Work on a small copy for speed
    scale = min(1.0, 300 / max(h, w))
    small = cv2.resize(background, None, fx=scale, fy=scale,
                       interpolation=cv2.INTER_AREA).astype(np.float32)
    sh, sw = small.shape

    # 2. Fit a smooth degree-2 surface (shadows / uneven lighting).
    #    Iteratively ignore dark pixels so stains don't pull the fit down.
    yy, xx = np.mgrid[0:sh, 0:sw]
    x, y, z = xx.ravel() / sw, yy.ravel() / sh, small.ravel()
    A = np.stack([x**i * y**j for i in range(3) for j in range(3 - i)], axis=1)
    keep = np.ones_like(z, bool)
    for _ in range(4):
        coef, *_ = np.linalg.lstsq(A[keep], z[keep], rcond=None)
        resid = z - A @ coef
        keep = resid > -2 * resid[keep].std()
    fit = (A @ coef).reshape(sh, sw)

    # 3. Residual relative to paper brightness: stains show up as negative blobs
    paper = max(np.median(fit), 1.0)
    rel = (small - fit) / paper
    blotchiness = float(rel.std())

    # 4. Candidate mask: noticeably darker than the fitted paper
    cand = (rel < -dark_thresh).astype(np.uint8) * 255
    cand = cv2.morphologyEx(cand, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))

    # 5. Colour of the paper background (LAB). Water/foxing stains are brown, i.e.
    #    redder (higher a*) and yellower (higher b*) than clean paper. The quadratic fit
    #    overshoots in the corners and the page edge is dark, but those regions keep the
    #    paper's colour, so colour separates real stains from these artefacts.
    if image.ndim == 3:
        colour_bg = cv2.medianBlur(cv2.dilate(image, kernel), window)
        lab = cv2.cvtColor(cv2.resize(colour_bg, (sw, sh), interpolation=cv2.INTER_AREA),
                           cv2.COLOR_BGR2LAB).astype(np.float32)
        clean = cand == 0
        a_shift = lab[:, :, 1] - np.median(lab[:, :, 1][clean])
        b_shift = lab[:, :, 2] - np.median(lab[:, :, 2][clean])
    else:
        a_shift = b_shift = None

    # 6. Keep only large blobs (stains are big, text residue is tiny) that are not
    #    page-edge strips and, for colour images, are redder than the paper.
    n, labels, stats, _ = cv2.connectedComponentsWithStats(cand)
    min_blob = 0.005 * sh * sw
    kept, redness_vals, yellow_vals = [], [], []
    for i in range(1, n):
        x0, y0, bw, bh, area = stats[i]
        if area < min_blob:
            continue
        touches_border = x0 == 0 or y0 == 0 or x0 + bw == sw or y0 + bh == sh
        thickness = area / max(bw, bh)       # mean width of the blob across its long axis
        thin = thickness < 0.05 * min(sh, sw)
        if touches_border and thin:          # dark page edge / scan border
            continue
        sel = labels == i
        if a_shift is not None:
            red = float(a_shift[sel].mean())
            if red < min_redness:            # same colour as paper -> lighting artefact
                continue
            redness_vals.append((red, area))
            yellow_vals.append((float(b_shift[sel].mean()), area))
        kept.append(i)

    stain_small = np.isin(labels, kept).astype(np.uint8) * 255
    areas = stats[kept, cv2.CC_STAT_AREA] if kept else np.array([])
    area_fraction = float(areas.sum() / (sh * sw))

    def wmean(vals):
        return float(sum(v * a for v, a in vals) / sum(a for _, a in vals)) if vals else 0.0

    mask = cv2.resize(stain_small, (w, h), interpolation=cv2.INTER_LINEAR)
    mask = (mask > 127).astype(np.uint8) * 255

    return {
        "blotchiness": blotchiness,
        "area_fraction": area_fraction,
        "num_large_blobs": int(len(kept)),
        "redness": wmean(redness_vals),      # mean LAB a* shift inside stains
        "yellowness": wmean(yellow_vals),    # mean LAB b* shift inside stains
        "mask": mask,
    }


def is_stain(image: np.ndarray,
             blotch_thresh: float = 0.03,
             area_thresh: float = 0.01) -> tuple[bool, dict]:
    """True if the image looks stained. Calibrate the thresholds on your own images."""
    f = stain_features(image)
    stained = f["blotchiness"] > blotch_thresh and f["area_fraction"] > area_thresh
    return stained, f


def show_stain(image: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Overlay the stain mask in red with its outline, for checking the detection."""
    overlay = image.copy()
    red = np.zeros_like(image); red[:, :, 2] = 255
    overlay[mask > 0] = cv2.addWeighted(image, 0.6, red, 0.4, 0)[mask > 0]
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(overlay, contours, -1, (0, 0, 255), 2)
    return overlay


# ---------- Run on Zoernikav folio212 ----------
img = cv2.imread("StainImages/Zoernikav folio212.jpg")
stained, f = is_stain(img)
print(f"stain={stained}  blotch={f['blotchiness']:.4f}  area={f['area_fraction']:.4f}  "
      f"blobs={f['num_large_blobs']}  red={f['redness']:.1f}  yellow={f['yellowness']:.1f}")
cv2.imwrite("StainImages/Zoernikav folio212_stainmask.png", f["mask"])
cv2.imwrite("StainImages/Zoernikav folio212_stainoverlay.png", show_stain(img, f["mask"]))


# ---------- Calibration: run on your labelled folders and compare the scores ----------
# from pathlib import Path
# for label in ["stain", "bleedthrough", "blur"]:
#     for p in sorted(Path(f"samples/{label}").glob("*")):
#         img = cv2.imread(str(p))
#         if img is None:
#             continue
#         stained, f = is_stain(img)
#         print(f"{label:13s} {p.name:30s} stain={stained}  "
#               f"blotch={f['blotchiness']:.4f}  area={f['area_fraction']:.4f}  "
#               f"blobs={f['num_large_blobs']}  red={f['redness']:.1f}  yellow={f['yellowness']:.1f}")